In [1]:
pip install langchain-core langchain-google-genai langgraph sentence-transformers neo4j langchain-community numpy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 37.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 45.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 463.6/463.6 kB 27.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 63.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 4.3 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: pydantic-core
    Found existing installation: pydantic_core 2.41.4
    Uninstalling pydantic_core-2.41.4:
      Successfully uninstalled pydantic_core-2.41.4
  Attempting uninstall: pyda

In [2]:
pip install langchain-groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 4.4 MB/s eta 0:00:00


In [3]:
pip install neo4j

In [4]:
!pip install ipython-autotime
%load_ext autotime

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 18.6 MB/s eta 0:00:00
time: 389 µs (started: 2026-03-21 11:04:59 +00:00)


In [5]:
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import interrupt
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_groq import ChatGroq
from sentence_transformers import SentenceTransformer
from neo4j import GraphDatabase
from typing import TypedDict, Annotated
import numpy as np
import uuid
import os

time: 39.1 s (started: 2026-03-21 11:04:59 +00:00)


In [6]:
embedder = SentenceTransformer("all-MiniLM-L6-v2")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

time: 6.97 s (started: 2026-03-21 11:05:38 +00:00)


In [7]:
from google.colab import userdata
api_go = userdata.get("Gemini")
api_gr = userdata.get("groqAPi")

time: 1.86 s (started: 2026-03-21 11:05:45 +00:00)


In [8]:
conv = ChatGoogleGenerativeAI(
    model = "gemini-2.5-flash-lite",
    temperature = 0.2,
    google_api_key = api_go
)

speed = ChatGroq(
    model = "llama-3.3-70b-versatile",
    temperature = 0,
    groq_api_key = api_gr
)

time: 1.24 s (started: 2026-03-21 11:05:47 +00:00)


### Relevancy tool


In [9]:
@tool
def relevancy_ebd(player_message: str, recent_context: str) -> str:
    """
  When to call: before passing any player message to an NPC or
                taking any game action

  player_message: the exact text the player just sent

  recent_context: one sentence summarising the last 2 exchanges,
                  or "no prior context" if this is the first message

  output: yes if the player message is relevant to the recent context,
          no otherwise
    """
    embedding_player = embedder.encode(player_message)
    embedding_context = embedder.encode(recent_context)
    dot= np.dot(embedding_player, embedding_context) / (np.linalg.norm(embedding_player) * np.linalg.norm(embedding_context))
    if dot > 0.5:
        return f"yes \n dot value: {dot}"
    else:
        return f"no \n dot value: {dot}"




time: 14.5 ms (started: 2026-03-21 11:05:48 +00:00)


In [10]:
@tool
def relevancy_llm(player_message: str, recent_context: str) -> str:
    """
  When to call: before passing any player message to an NPC or
                taking any game action

  player_message: the exact text the player just sent

  recent_context: one sentence summarising the last 2 exchanges,
                  or "no prior context" if this is the first message

  output: yes if the player message is relevant to the recent context,
          no otherwise
    """
    prompt = f"""You are a very experienced context manager
    you have to figure out based on previous discussion of the player with npc whether the current message is relevant or the player is just messing about

    here is the player message: {player_message}

    here is the recent context: {recent_context}

    STRICT OUTPUT CONDITION: only "yes" if the player message is relevant to the recent context,
          "no" otherwise, no justification or any other text
    """
    return speed.invoke(prompt)


time: 11.4 ms (started: 2026-03-21 11:05:48 +00:00)


In [11]:
player_message = "We saw the missing page near the fireplace half burnt, it was near the table you were studying"
recent_context = "Where were you?\nI was in the study room\nDo you know about page 42?\nno, i saw the missing page when i entered thorne's room"

time: 723 µs (started: 2026-03-21 11:05:48 +00:00)


In [12]:
%%time
ans = relevancy_llm.invoke({"player_message": player_message, "recent_context" : recent_context})
print(ans.content)

yes
CPU times: user 62.6 ms, sys: 7.96 ms, total: 70.6 ms
Wall time: 409 ms
time: 410 ms (started: 2026-03-21 11:05:48 +00:00)


In [13]:
%%time
ans = ans = relevancy_ebd.invoke({"player_message": player_message, "recent_context" : recent_context})
print(ans)

yes 
 dot value: 0.6400977969169617
CPU times: user 101 ms, sys: 10.5 ms, total: 112 ms
Wall time: 556 ms
time: 561 ms (started: 2026-03-21 11:05:49 +00:00)


### Summary

In [15]:
import json
import os

@tool
def running_summ(npc_name: str, old_summary: str, question: str, answer: str) -> str:
  """"
  summarises the current message into a running summary of messages conversed with the npc
  """
  prompt = f"""Update this NPC memory state by merging the new interaction.
  Output one concise paragraph (max 4 sentences), past tense, third person.

  STATE: {old_summary or 'none'}
  Q: {question}
  A: {answer}
  NPC: {npc_name}

  Updated state:
  """
  summ = speed.invoke(prompt)
  return summ.content




time: 7.32 ms (started: 2026-03-21 11:09:04 +00:00)


### TO DB

In [19]:
@tool
def send_to_db(npc_id: str, npc_name: str, player_id: str, summary: str) -> dict:
  """"
  send the summary to the db
  """
  payload = {
        "npc_id": npc_id,
        "npc_name": npc_name,
        "player_id": player_id,
        "summary": summary
    }

  print(json.dumps(payload, indent=2))
  return {"status": "success", "npc_id": npc_id}


time: 7.75 ms (started: 2026-03-21 11:09:39 +00:00)


### DB

In [20]:
from neo4j import GraphDatabase

time: 563 µs (started: 2026-03-21 11:09:42 +00:00)


In [21]:
driver = GraphDatabase.driver(
    "bolt://localhost:7687",
    auth=("neo4j", "neo4j123")
)

time: 1.32 ms (started: 2026-03-21 11:09:42 +00:00)


In [22]:
def setup_db():
    with driver.session() as session:
        session.run("""
        CREATE CONSTRAINT cache_key IF NOT EXISTS
        FOR (c:Cache) REQUIRE c.key IS UNIQUE
        """)
        session.run("""
        MERGE (a:NPC {name:'arjun'})
        SET a.personality = 'nervous, polite, avoids conflict',
            a.truth = 'I took page 42. Thorne was alive.',
            a.lie = 'The page was damaged',
            a.state = 'COMPOSED'
        """)
        session.run("""
        MERGE (b:NPC {name:'bell'})
        SET b.personality = 'arrogant, intellectual',
            b.truth = 'I visited storage and saw Graves',
            b.lie = 'I never left the hall',
            b.state = 'COMPOSED'
        """)
        session.run("""
        MERGE (g:NPC {name:'graves'})
        SET g.personality = 'calm, controlled',
            g.truth = 'I poisoned the brandy',
            g.lie = 'I stayed in office',
            g.state = 'COMPOSED'
        """)
        session.run("""
        MERGE (e:Evidence {id:'page_42'})
        SET e.description = 'Torn manuscript page',
            e.discovered = false
        """)

time: 961 µs (started: 2026-03-21 11:09:43 +00:00)


In [26]:
import hashlib
from typing import TypedDict, List, Dict

time: 841 µs (started: 2026-03-21 11:11:01 +00:00)


In [27]:
class GameState(TypedDict):
    player_input: str
    chat_history: List[Dict]
    npc_states: Dict
    evidence: List[str]
    active_npc: str


def get_npc_data(npc):
    with driver.session() as session:
        result = session.run("""
        MATCH (n:NPC {name:$name})
        RETURN n.personality AS personality,
               n.truth AS truth,
               n.lie AS lie,
               n.state AS state
        """, name=npc).single()
        if result is None:
            return {"personality": "unknown", "truth": "", "lie": "", "state": "COMPOSED"}
        return dict(result)



def get_evidence():
    with driver.session() as session:
        result = session.run("""
        MATCH (e:Evidence)
        WHERE e.discovered = true
        RETURN e.id AS id
        """)
        return [r["id"] for r in result]


def update_npc_state(npc, user_input, evidence):
    text = user_input.lower()
    if npc == 'arjun':
        if 'page 42' in text: return 'DEFENSIVE'
        if 'show page 42' in text: return 'BREAKDOWN'
    if npc == 'bell':
        if 'storage' in text: return 'DEFENSIVE'
        if 'vial' in text: return 'BREAKDOWN'
    if npc == 'graves':
        if 'ledger' in text: return 'DEFENSIVE'
        if all(x in text for x in ['ledger','pantry','vial']): return 'BREAKDOWN'
    return 'COMPOSED'



def get_cache(prompt):
    key = hashlib.md5(prompt.encode()).hexdigest()
    with driver.session() as session:
        res = session.run("MATCH (c:Cache {key:$key}) RETURN c.response AS response", key=key).single()
        return res['response'] if res else None

def set_cache(prompt, response):
    key = hashlib.md5(prompt.encode()).hexdigest()
    with driver.session() as session:
        session.run("MERGE (c:Cache {key:key}) SET c.response= response", key=key, response=response)


def build_prompt(npc, state, user_input):
    data = get_npc_data(npc)
    evidence = get_evidence()
    mental = state['npc_states'][npc]
    system_prompt = f"""
You are {npc}, a murder suspect. Stay in character. Max 2 sentences.
Personality: {data['personality']}
Truth: {data['truth']}
Lie: {data['lie']}
Mental State: {mental}
Evidence: {evidence}
"""
    return [
        {'role':'system','content':system_prompt},
        {'role':'user','content':user_input}
    ]

def clean_response(text):
    text = text.strip().split("\n")[0]
    for p in ['arjun:','player:','assistant:','npc:']:
        if text.lower().startswith(p):
            text = text[len(p):].strip()
    return text



def npc_response(state):
    npc = state['active_npc']
    user_input = state['player_input']
    new_state = update_npc_state(npc, user_input, state['evidence'])
    state['npc_states'][npc] = new_state
    messages = build_prompt(npc, state, user_input)
    cache_key = ''.join([m['content'] for m in messages])
    cached = get_cache(cache_key)
    if cached:
        response_text = cached
    else:
        response = llm.invoke(messages)
        response_text = clean_response(response.content)
        set_cache(cache_key, response_text)
    state['chat_history'].append({
        'npc': npc,
        'state': new_state,
        'text': response_text
    })
    return state


def unlock_evidence(evidence_id):
    with driver.session() as session:
        session.run("MATCH (e:Evidence {id:$id}) SET e.discovered=true", id=evidence_id)


def route_npc(user_input):
    text = user_input.lower()
    if 'arjun' in text: return 'arjun'
    if 'bell' in text: return 'bell'
    if 'graves' in text: return 'graves'
    return 'arjun'



def run_game():
    state = {
        'player_input':'',
        'chat_history':[],
        'npc_states':{'arjun':'COMPOSED','bell':'COMPOSED','graves':'COMPOSED'},
        'evidence':[],
        'active_npc':'arjun'
    }
    print("\n🕵️ Welcome to The Shimla Ledger")
    print("Type 'talk arjun', 'talk bell', 'talk graves' to switch NPC")
    print("Type 'exit' to quit")
    while True:
        user_input = input("\nYou: ")
        if user_input.lower() == 'exit': break
        if user_input.startswith('talk'):
            parts = user_input.split(' ')
            if len(parts) > 1:
                state['active_npc'] = parts[1]
                print(f"\n➡️ Now talking to {parts[1]}")
            else:
                print("⚠️ Specify NPC name")
            continue
        state['player_input'] = user_input
        state = npc_response(state)
        last = state['chat_history'][-1]
        print(f"\n{last['npc']} ({last['state']}): {last['text']}")


setup_db()
run_game()


ServiceUnavailable: Couldn't connect to localhost:7687 (resolved to ('[::1]:7687', '127.0.0.1:7687')):
Failed to establish connection to ResolvedIPv6Address(('::1', 7687, 0, 0)) (reason [Errno 111] Connection refused)
Failed to establish connection to ResolvedIPv4Address(('127.0.0.1', 7687)) (reason [Errno 111] Connection refused)

time: 49.9 ms (started: 2026-03-21 11:11:05 +00:00)


time: 144 ms (started: 2026-03-21 11:10:02 +00:00)
